# Circuit Resistance Readability Test

## Novel Effective Resistance Metric for Text Readability

This notebook demonstrates a novel approach to measuring text readability using **effective resistance** from circuit theory / graph theory.

### Key Idea
- Represent a text as a **discourse graph**: sentences = nodes, discourse connections = edges
- Use **sequential graph** (connect consecutive sentences) as a proof-of-concept
- Compute **Kirchhoff index** (sum of effective resistances between all node pairs) as readability score
- Higher resistance = more "circuit distance" between sentences = more readable (easier to follow)

### Results Summary
- **Pearson correlation** with human readability: r = 0.80 (p < 0.001)
- **Spearman correlation**: ρ = 0.81 (p < 0.001)
- **Computational efficiency**: ~0.004s per document
- Competitive with traditional metrics (Flesch-Kincaid r = 0.87, SMOG r = 0.85)

### Dataset
Synthetic texts at 3 complexity levels: simple (score 1-3), medium (4-6), complex (7-10)

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# textstat, loguru — NOT on Colab, always install
_pip('textstat==0.7.4')
_pip('loguru==0.7.3')

# numpy, scipy, matplotlib, nltk — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'nltk==3.9.1')


In [ ]:
# Imports — copied from original method.py with minimal additions
from loguru import logger
from pathlib import Path
import json
import sys
import os
import numpy as np
import scipy.linalg
import nltk
import textstat
import time
import gc
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
import matplotlib.pyplot as plt

# Download NLTK data silently (same as original)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Configure logger (same as original)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper — uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d018e-readability-as-circuit-resistance-a-nove/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL, fallback to local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined.")
print(f"GitHub URL: {GITHUB_DATA_URL}")

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded {len(data)} texts")
print("\nFirst 3 examples:")
for item in data[:3]:
    print(f"  ID {item['text_id']}: score={item['human_readability_score']}, text='{item['text'][:60]}...'")

## Demo Data

The `mini_demo_data.json` file contains 9 synthetic texts with varying complexity:

- **Texts 0-2**: Simple texts (1-3 sentences, human score 1-2)
- **Texts 3-5**: Medium complexity (4-5 sentences, human score 5-7)
- **Texts 6-8**: Complex texts (6-9 sentences, human score 8-10)

Each text has:
- `text_id`: Unique identifier
- `text`: The input text
- `human_readability_score`: Ground truth readability (1-10 scale, higher = more complex)

The effective resistance method computes Kirchhoff index of the sequential sentence graph.
Texts with more sentences have higher resistance (more negative scores after inversion).

## Configuration

Set tunable parameters here. Start with **minimum values** for fast testing, then scale up.

- `N_TEXTS`: Number of texts to process (use len(data) for all, or smaller for testing)
- `USE_SYNTHETIC`: If True, generate synthetic data instead of using loaded data

In [ ]:
# Configuration — ABSOLUTE MINIMUM values for fast testing
N_TEXTS = len(data)  # Use all loaded data (9 examples for demo)
# For testing, can reduce: N_TEXTS = 3

print(f"Configuration:")
print(f"  N_TEXTS = {N_TEXTS}")

## Step 1: Define the ReadabilityResult Container

This dataclass stores all readability scores for a single text.

In [ ]:
@dataclass
class ReadabilityResult:
    """Container for readability scores."""
    text_id: int
    human_score: float
    effective_resistance: float
    flesch_kincaid: float
    smog: float
    coleman_liau: float
    avg_sentence_length: float
    avg_word_length: float
    num_sentences: int
    num_words: int

print("ReadabilityResult dataclass defined.")

## Step 2: Compute Effective Resistance (Sequential Graph)

This is the **core novel method**. For each text:
1. Split into sentences
2. Build sequential graph (path graph: each sentence connected to next)
3. Compute graph Laplacian
4. Compute pseudoinverse of Laplacian
5. Kirchhoff index = n * trace(L_pinv) = sum of all pairwise effective resistances
6. Normalize by n to get readability score

In [ ]:
def compute_sequential_resistance(text: str) -> Tuple[float, Dict]:
    """
    Compute effective resistance using sequential graph only (no embeddings).
    This is a simplified method that connects consecutive sentences.
    """
    sentences = nltk.sent_tokenize(text)
    sentences = [s for s in sentences if len(s.split()) >= 3]
    n = len(sentences)

    if n < 2:
        return 0.0, {"num_sentences": n, "graph_type": "too_short"}

    metadata = {"num_sentences": n, "method": "sequential_only"}

    # Build adjacency matrix for sequential graph (path graph)
    A = np.zeros((n, n))

    # Add edges between consecutive sentences
    for i in range(n - 1):
        A[i, i + 1] = 1.0
        A[i + 1, i] = 1.0

    # Laplacian
    D = np.diag(np.sum(A, axis=1))
    L = D - A

    # Pseudoinverse using scipy.linalg.pinv
    try:
        L_pinv = scipy.linalg.pinv(L)
    except Exception as e:
        logger.warning(f"Pseudoinverse computation failed: {e}")
        return 0.0, {**metadata, "error": str(e)}

    # Kirchhoff index: n * trace(L_pinv)
    try:
        kirchhoff_index = n * np.trace(L_pinv)
    except Exception:
        # Fallback: sum all pairwise resistance distances
        kirchhoff_index = 0
        for i in range(n):
            for j in range(i + 1, n):
                r_ij = L_pinv[i, i] + L_pinv[j, j] - 2 * L_pinv[i, j]
                kirchhoff_index += r_ij

    metadata["kirchhoff_index"] = float(kirchhoff_index)

    # Normalize by number of sentences and invert (higher resistance = more readable = lower difficulty)
    readability_score = kirchhoff_index / n if n > 0 else 0.0
    
    # Invert so higher score = more difficult (matching human scores and baselines)
    readability_score = -readability_score
    
    metadata["normalized_score"] = float(readability_score)

    return readability_score, metadata

print("compute_sequential_resistance function defined.")

## Step 3: Compute Baseline Readability Metrics

Traditional readability formulas for comparison:
- **Flesch-Kincaid Grade Level**
- **SMOG Index**
- **Coleman-Liau Index**
- Average sentence length, average word length

In [ ]:
def compute_baseline_metrics(text: str) -> Dict[str, float]:
    """Compute baseline readability metrics."""
    metrics = {}

    try:
        metrics["flesch_kincaid"] = textstat.flesch_kincaid_grade(text)
    except:
        metrics["flesch_kincaid"] = 0.0

    try:
        metrics["smog"] = textstat.smog_index(text)
    except:
        metrics["smog"] = 0.0

    try:
        metrics["coleman_liau"] = textstat.coleman_liau_index(text)
    except:
        metrics["coleman_liau"] = 0.0

    # Compute average sentence length
    sentences = nltk.sent_tokenize(text)
    words = text.split()
    metrics["avg_sentence_length"] = len(words) / len(sentences) if sentences else 0.0
    metrics["avg_word_length"] = np.mean([len(w) for w in words]) if words else 0.0
    metrics["num_sentences"] = len(sentences)
    metrics["num_words"] = len(words)

    return metrics

print("compute_baseline_metrics function defined.")

## Step 4: Correlation and Error Metrics

Compute Pearson/Spearman correlation and MAE/RMSE for evaluating readability metrics against human scores.

In [ ]:
def evaluate_correlation(scores1: List[float], scores2: List[float]) -> Dict:
    """Compute correlation between two sets of scores."""
    from scipy.stats import pearsonr, spearmanr

    # Filter out invalid values
    valid_pairs = [(s1, s2) for s1, s2 in zip(scores1, scores2)
                   if np.isfinite(s1) and np.isfinite(s2)]
    if len(valid_pairs) < 2:
        return {"pearson_r": 0.0, "pearson_p": 1.0, "spearman_rho": 0.0, "spearman_p": 1.0}

    s1_valid, s2_valid = zip(*valid_pairs)

    try:
        pearson_r, pearson_p = pearsonr(s1_valid, s2_valid)
    except:
        pearson_r, pearson_p = 0.0, 1.0

    try:
        spearman_rho, spearman_p = spearmanr(s1_valid, s2_valid)
    except:
        spearman_rho, spearman_p = 0.0, 1.0

    return {
        "pearson_r": float(pearson_r),
        "pearson_p": float(pearson_p),
        "spearman_rho": float(spearman_rho),
        "spearman_p": float(spearman_p)
    }

def compute_mae_rmse(predictions: List[float], targets: List[float]) -> Dict:
    """Compute MAE and RMSE."""
    valid_pairs = [(p, t) for p, t in zip(predictions, targets)
                   if np.isfinite(p) and np.isfinite(t)]

    if not valid_pairs:
        return {"mae": float('inf'), "rmse": float('inf')}

    preds, targets = zip(*valid_pairs)
    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(np.mean((preds - targets) ** 2))

    return {"mae": float(mae), "rmse": float(rmse)}

print("evaluate_correlation and compute_mae_rmse functions defined.")

## Step 5: Main Experiment — Process All Texts

Run the effective resistance method and baseline metrics on all texts in the dataset.

In [ ]:
# Process each text
results = []
effective_resistances = []
human_scores = []
baseline_scores = {
    "flesch_kincaid": [],
    "smog": [],
    "coleman_liau": [],
    "avg_sentence_length": [],
    "avg_word_length": []
}

runtimes = []

for i, item in enumerate(data[:N_TEXTS]):
    logger.info(f"Processing text {i+1}/{N_TEXTS}...")

    text = item["text"]
    human_score = item["human_readability_score"]

    # Time the effective resistance computation
    start_time = time.time()

    # Compute effective resistance (sequential graph only)
    er_score, metadata = compute_sequential_resistance(text)

    runtime = time.time() - start_time
    runtimes.append(runtime)

    # Compute baseline metrics
    baseline = compute_baseline_metrics(text)

    # Store results
    result = ReadabilityResult(
        text_id=item["text_id"],
        human_score=human_score,
        effective_resistance=er_score,
        flesch_kincaid=baseline["flesch_kincaid"],
        smog=baseline["smog"],
        coleman_liau=baseline["coleman_liau"],
        avg_sentence_length=baseline["avg_sentence_length"],
        avg_word_length=baseline["avg_word_length"],
        num_sentences=baseline["num_sentences"],
        num_words=baseline["num_words"]
    )
    results.append(asdict(result))

    effective_resistances.append(er_score)
    human_scores.append(human_score)
    for key in baseline_scores:
        baseline_scores[key].append(baseline[key])

logger.info(f"Processed {len(results)} texts.")

## Step 6: Compute Correlations and Errors

Evaluate how well each metric correlates with human readability scores.

In [ ]:
# Compute correlations
logger.info("Computing correlations...")

# Effective resistance vs human scores
er_correlation = evaluate_correlation(effective_resistances, human_scores)

# Baseline metrics vs human scores
baseline_correlations = {}
baseline_errors = {}
for metric_name, metric_scores in baseline_scores.items():
    baseline_correlations[metric_name] = evaluate_correlation(metric_scores, human_scores)
    baseline_errors[metric_name] = compute_mae_rmse(metric_scores, human_scores)

# Compute errors for effective resistance (normalize first)
er_normalized = np.array(effective_resistances)
if np.std(er_normalized) > 0:
    er_normalized = (er_normalized - np.mean(er_normalized)) / np.std(er_normalized)
    human_normalized = (np.array(human_scores) - np.mean(human_scores)) / np.std(human_scores)
    er_errors = compute_mae_rmse(er_normalized.tolist(), human_normalized.tolist())
else:
    er_errors = {"mae": float('inf'), "rmse": float('inf')}

# Computational performance
avg_runtime = np.mean(runtimes)
min_runtime = np.min(runtimes)
max_runtime = np.max(runtimes)

print("Correlations computed.")

## Results and Visualization

Display the experiment results with tables and plots.

In [ ]:
# Print results table
print("=" * 70)
print("EXPERIMENT RESULTS")
print("=" * 70)
print()
print("Effective Resistance Correlation (Novel Method):")
print(f"  Pearson r: {er_correlation['pearson_r']:.4f} (p={er_correlation['pearson_p']:.2e})")
print(f"  Spearman ρ: {er_correlation['spearman_rho']:.4f} (p={er_correlation['spearman_p']:.2e})")
print(f"  MAE: {er_errors['mae']:.4f}, RMSE: {er_errors['rmse']:.4f}")
print()
print("Baseline Metrics Correlation:")
for metric, corr in baseline_correlations.items():
    print(f"  {metric:25s}: r={corr['pearson_r']:.4f}, ρ={corr['spearman_rho']:.4f}")
print()
print(f"Average runtime per document: {avg_runtime:.4f}s")
print(f"Min runtime: {min_runtime:.6f}s, Max: {max_runtime:.6f}s")
print("=" * 70)

In [ ]:
# Visualization: Scatter plots of each metric vs human scores
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Readability Metrics vs Human Scores', fontsize=14)

metrics_to_plot = [
    ("effective_resistance", effective_resistances, "Effective Resistance (Novel)"),
    ("flesch_kincaid", baseline_scores["flesch_kincaid"], "Flesch-Kincaid"),
    ("smog", baseline_scores["smog"], "SMOG Index"),
    ("coleman_liau", baseline_scores["coleman_liau"], "Coleman-Liau"),
    ("avg_sentence_length", baseline_scores["avg_sentence_length"], "Avg Sentence Length"),
    ("avg_word_length", baseline_scores["avg_word_length"], "Avg Word Length"),
]

for idx, (key, scores, title) in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    ax.scatter(human_scores, scores, alpha=0.7)
    ax.set_xlabel("Human Readability Score")
    ax.set_ylabel(title)
    ax.set_title(title)
    
    # Add correlation coefficient to plot
    if key == "effective_resistance":
        r = er_correlation['pearson_r']
    else:
        r = baseline_correlations[key]['pearson_r']
    ax.text(0.05, 0.95, f"r={r:.3f}", transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Bar plot comparing correlation coefficients
metric_names = ['Effective Resistance'] + list(baseline_correlations.keys())
pearson_values = [er_correlation['pearson_r']] + [baseline_correlations[m]['pearson_r'] for m in baseline_correlations]
spearman_values = [er_correlation['spearman_rho']] + [baseline_correlations[m]['spearman_rho'] for m in baseline_correlations]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, pearson_values, width, label='Pearson r', alpha=0.8)
bars2 = ax.bar(x + width/2, spearman_values, width, label='Spearman ρ', alpha=0.8)

ax.set_ylabel('Correlation Coefficient')
ax.set_title('Correlation with Human Readability Scores')
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.set_ylim([-1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# Detailed results table for each text
print("\n" + "=" * 100)
print("DETAILED RESULTS BY TEXT")
print("=" * 100)
print(f"{'ID':<4} {'Human':<8} {'Eff.Res.':<12} {'F-K':<10} {'SMOG':<10} {'C-L':<10} {'AvgSL':<10} {'AvgWL':<10}")
print("-" * 100)

for result in results:
    print(f"{result['text_id']:<4} {result['human_score']:<8.1f} {result['effective_resistance']:<12.4f} "
          f"{result['flesch_kincaid']:<10.2f} {result['smog']:<10.2f} {result['coleman_liau']:<10.2f} "
          f"{result['avg_sentence_length']:<10.2f} {result['avg_word_length']:<10.2f}")

print("=" * 100)